<a href="https://www.kaggle.com/code/avikdas567/deep-past-challenge-translate-akkadian-to-english?scriptVersionId=292053675" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/deep-past-initiative-machine-translation/sample_submission.csv
/kaggle/input/deep-past-initiative-machine-translation/bibliography.csv
/kaggle/input/deep-past-initiative-machine-translation/publications.csv
/kaggle/input/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv
/kaggle/input/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv
/kaggle/input/deep-past-initiative-machine-translation/eBL_Dictionary.csv
/kaggle/input/deep-past-initiative-machine-translation/train.csv
/kaggle/input/deep-past-initiative-machine-translation/test.csv
/kaggle/input/deep-past-initiative-machine-translation/published_texts.csv
/kaggle/input/deep-past-initiative-machine-translation/resources.csv


In [2]:
# ============================================================
# Deep Past Initiative - Akkadian → English
# ============================================================

import re
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---------------------------
# Configuration
# ---------------------------
DATA_DIR = "/kaggle/input/deep-past-initiative-machine-translation"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_SRC_LEN = 300
MAX_TGT_LEN = 300
BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-3

print("Device:", DEVICE)

# ---------------------------
# Load data
# ---------------------------
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")

# ---------------------------
# Cleaning
# ---------------------------
def clean_translit(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"[!?/:]", " ", text)
    text = re.sub(r"[\[\]˹˺]", "", text)
    text = re.sub(r"\b\d+'*\b", " ", text)
    text = re.sub(r"\.{2,}", "<big_gap>", text)
    text = re.sub(r"\bx\b", "<gap>", text)
    text = re.sub(r"\{[^}]+\}", "<det>", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["src"] = train_df["transliteration"].apply(clean_translit)
train_df["tgt"] = train_df["translation"].astype(str).str.lower()
test_df["src"]  = test_df["transliteration"].apply(clean_translit)

# ---------------------------
# Character vocab
# ---------------------------
SPECIALS = ["<pad>", "<sos>", "<eos>"]
chars = set()

for s in train_df["src"].tolist() + train_df["tgt"].tolist():
    chars.update(list(s))

itos = SPECIALS + sorted(chars)
stoi = {c: i for i, c in enumerate(itos)}

PAD, SOS, EOS = 0, 1, 2
VOCAB_SIZE = len(itos)

print("Vocab size:", VOCAB_SIZE)

# ---------------------------
# Dataset
# ---------------------------
class CharDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df
        self.train = train

    def encode(self, text, max_len):
        ids = [SOS] + [stoi.get(c, PAD) for c in text[:max_len-2]] + [EOS]
        ids += [PAD] * (max_len - len(ids))
        return torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        src = self.encode(self.df.iloc[idx]["src"], MAX_SRC_LEN)
        if self.train:
            tgt = self.encode(self.df.iloc[idx]["tgt"], MAX_TGT_LEN)
            return src, tgt
        return src

# ---------------------------
# Model
# ---------------------------
class CharSeq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE, 128, padding_idx=PAD)
        self.enc = nn.GRU(128, 256, batch_first=True)
        self.dec = nn.GRU(128, 256, batch_first=True)
        self.out = nn.Linear(256, VOCAB_SIZE)

    def forward(self, src, tgt):
        src_emb = self.emb(src)
        _, h = self.enc(src_emb)
        tgt_emb = self.emb(tgt[:, :-1])
        out, _ = self.dec(tgt_emb, h)
        return self.out(out)

# ---------------------------
# Train
# ---------------------------
train_ds = CharDataset(train_df, train=True)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

model = CharSeq2Seq().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

for epoch in range(EPOCHS):
    model.train()
    total = 0
    for src, tgt in train_dl:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        logits = model(src, tgt)
        loss = loss_fn(
            logits.reshape(-1, VOCAB_SIZE),
            tgt[:, 1:].reshape(-1)
        )
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} - loss {total/len(train_dl):.4f}")

# ---------------------------
# Decode
# ---------------------------
def decode(src):
    src = src.unsqueeze(0).to(DEVICE)
    src_emb = model.emb(src)
    _, h = model.enc(src_emb)

    cur = torch.tensor([[SOS]], device=DEVICE)
    out = []

    for _ in range(MAX_TGT_LEN):
        emb = model.emb(cur)
        o, h = model.dec(emb, h)
        token = o.argmax(-1).item()

        # <<< FIX >>>
        if token >= VOCAB_SIZE:
            break
        if token == EOS:
            break

        out.append(itos[token])
        cur = torch.tensor([[token]], device=DEVICE)

    return "".join(out)

# ---------------------------
# Inference
# ---------------------------
test_ds = CharDataset(test_df, train=False)
model.eval()

translations = []
for i in tqdm(range(len(test_ds))):
    translations.append(decode(test_ds[i]))

# ---------------------------
# Submission
# ---------------------------
submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": translations
})

submission.to_csv("submission.csv", index=False)
print("submission.csv saved")
submission.head()

Device: cuda
Vocab size: 100
Epoch 1/10 - loss 3.3324
Epoch 2/10 - loss 2.5042
Epoch 3/10 - loss 2.2213
Epoch 4/10 - loss 2.0006
Epoch 5/10 - loss 1.8180
Epoch 6/10 - loss 1.6775
Epoch 7/10 - loss 1.5659
Epoch 8/10 - loss 1.4787
Epoch 9/10 - loss 1.4086
Epoch 10/10 - loss 1.3467


100%|██████████| 4/4 [00:00<00:00, 28.31it/s]

submission.csv saved


,id,translation
0,0,
1,1,
2,2,
3,3,
